In [4]:
!pip install torch
!pip install transformers
!pip install pandas scikit-learn


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 96.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 77.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 41.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 13.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 40.0 MB/s eta 0:00:00
  Attempting uninstall: nvidia-nvjitlink-cu12
    Found existing installation: nvidia-nvjitlink-cu12 12.5.82
    Uninstalling nvidia-nvjitlink-cu12-12.5.82:
      Successfully uninstalled nvidia-nvjitlin

In [6]:
!pip install torch transformers pandas scikit-learn


In [9]:
from google.colab import files
uploaded = files.upload()

Saving feedback_log.csv to feedback_log.csv


In [11]:
df = pd.read_csv("feedback_log.csv")
print(df.head())

  You are a helpful assistant. Use the following context to answer the question.\n\nContext:\nfor other people with the same name, see rob brown. robert william brown born april 10, 1968 is a canadian former professional ice hockey right winger. rob brown he is best known for his time spent playing for the pittsburgh penguins from his debut in 1987 until 1990, and then again from 1997 until 2000. between and following these stints, brown shuffled between minor league teams in the international hockey league ihl and other nhl teams, including the hartford whalers, chicago blackhawks, dallas stars, and los angeles kings. playing career as a youth, he played in the 1981 quebec international pee wee hockey tournament with a minor ice hockey team from oshawa. brown was a prolific scorer at the junior level, averaging over two points per game brown in 2010 during his junior career. in particular, brown flourished in 1986 87 winning multiple born april 10, 1968 age 57 awards including most va

In [14]:
df = pd.read_csv("feedback_log.csv", sep=",", quotechar='"')
print(df.head())


  You are a helpful assistant. Use the following context to answer the question.\n\nContext:\nfor other people with the same name, see rob brown. robert william brown born april 10, 1968 is a canadian former professional ice hockey right winger. rob brown he is best known for his time spent playing for the pittsburgh penguins from his debut in 1987 until 1990, and then again from 1997 until 2000. between and following these stints, brown shuffled between minor league teams in the international hockey league ihl and other nhl teams, including the hartford whalers, chicago blackhawks, dallas stars, and los angeles kings. playing career as a youth, he played in the 1981 quebec international pee wee hockey tournament with a minor ice hockey team from oshawa. brown was a prolific scorer at the junior level, averaging over two points per game brown in 2010 during his junior career. in particular, brown flourished in 1986 87 winning multiple born april 10, 1968 age 57 awards including most va

In [15]:
df_raw = pd.read_csv("feedback_log.csv", header=None, names=["raw"])
print(df_raw.head())


                                                 raw
0  You are a helpful assistant. Use the following...
1  Sentence: bob brown was a prolific scorer at t...
2  You are a helpful assistant. Use the following...
3  You are a helpful assistant. Use the following...
4  You are a helpful assistant. Use the following...


In [16]:
def split_row(row):
    raw = row['raw']
    # Split from right by the last comma to separate feedback label
    if ',' in raw:
        parts = raw.rsplit(',', 1)  # split into [prompt_and_answer, feedback]
        feedback = parts[1].strip()
        prompt_and_answer = parts[0].strip()

        # Now try splitting prompt and answer by 'Answer:' keyword
        if "Answer:" in prompt_and_answer:
            prompt_part, answer_part = prompt_and_answer.split("Answer:", 1)
        else:
            prompt_part = prompt_and_answer
            answer_part = ""

        return pd.Series({
            "prompt": prompt_part.strip(),
            "answer": answer_part.strip().strip('"').replace('\n',' '),
            "feedback": feedback
        })
    else:
        return pd.Series({
            "prompt": raw,
            "answer": "",
            "feedback": ""
        })

df_parsed = df_raw.apply(split_row, axis=1)
print(df_parsed.head())


                                              prompt  \
0  You are a helpful assistant. Use the following...   
1  Sentence: bob brown was a prolific scorer at t...   
2  You are a helpful assistant. Use the following...   
3  You are a helpful assistant. Use the following...   
4  You are a helpful assistant. Use the following...   

                                              answer     feedback  
0                                                      "bob brown  
1                                                     thumbs_down  
2  ,Michael Chang won the French Open at the age ...    thumbs_up  
3  ,Michael Chang won the French Open at the age ...    thumbs_up  
4                                                             "34  


In [17]:
df_parsed.to_csv("cleaned_feedback.csv", index=False)


In [21]:
import pandas as pd

df = pd.read_csv("cleaned_feedback.csv")  # Your cleaned CSV with proper columns

# Example: Convert feedback to numeric labels (1 for thumbs_up, 0 for thumbs_down)
df["label"] = df["feedback"].apply(lambda x: 1 if "up" in str(x).lower() else 0)


In [22]:
from torch.utils.data import Dataset

class FeedbackDataset(Dataset):
    def __init__(self, dataframe, tokenizer, max_length=512):
        self.data = dataframe
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        prompt = self.data.iloc[idx]["prompt"]
        response = self.data.iloc[idx]["response"]
        text = prompt + " " + response
        inputs = self.tokenizer(
            text,
            max_length=self.max_length,
            padding="max_length",
            truncation=True,
            return_tensors="pt"
        )
        label = self.data.iloc[idx]["label"]
        return {
            "input_ids": inputs["input_ids"].squeeze(0),
            "attention_mask": inputs["attention_mask"].squeeze(0),
            "labels": torch.tensor(label, dtype=torch.float)
        }


In [23]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

model_name = "bert-base-uncased"  # Or any other lightweight model you prefer

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=1)


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [31]:
print(df.columns)


Index(['prompt', 'answer', 'feedback'], dtype='object')


In [33]:
import pandas as pd
import torch
from torch.utils.data import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments,
)

# === Load CSV ===
df = pd.read_csv("cleaned_feedback.csv")

# === Create label column from feedback ===
# Label is 1 if 'up' in feedback, else 0
df["label"] = df["feedback"].apply(lambda x: 1 if isinstance(x, str) and "up" in x.lower() else 0)

# Optional: verify
print(df[["feedback", "label"]].head())

# === Define custom Dataset ===
class FeedbackDataset(Dataset):
    def __init__(self, dataframe, tokenizer, max_length=512):
        self.tokenizer = tokenizer
        self.data = dataframe
        self.max_length = max_length

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        prompt = str(self.data.iloc[idx]["prompt"])
        answer = str(self.data.iloc[idx]["answer"])
        label = int(self.data.iloc[idx]["label"])
        full_text = prompt + " " + answer

        encoding = self.tokenizer(
            full_text,
            truncation=True,
            padding="max_length",
            max_length=self.max_length,
            return_tensors="pt"
        )

        return {
            "input_ids": encoding["input_ids"].squeeze(0),
            "attention_mask": encoding["attention_mask"].squeeze(0),
            "labels": torch.tensor(label, dtype=torch.long)
        }

# === Load tokenizer and model ===
model_name = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)

# === Create dataset ===
dataset = FeedbackDataset(df, tokenizer)

# === Training arguments ===
training_args = TrainingArguments(
    output_dir="./reward_model",
    per_device_train_batch_size=8,
    num_train_epochs=3,
    logging_dir="./logs",
    logging_steps=10,
    save_strategy="no",
    report_to="none"
)

# === Trainer ===
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset
)

# === Train the model ===
trainer.train()

# === Save model and tokenizer ===
model.save_pretrained("reward_model")
tokenizer.save_pretrained("reward_model")
print("Model and tokenizer saved to ./reward_model")


      feedback  label
0   "bob brown      0
1  thumbs_down      0
2    thumbs_up      1
3    thumbs_up      1
4          "34      0


Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Step,Training Loss
10,0.704000
20,0.693700
30,0.665600
40,0.629900
50,0.578100


Model and tokenizer saved to ./reward_model


In [34]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch

# Load your reward model
tokenizer = AutoTokenizer.from_pretrained("reward_model")
model = AutoModelForSequenceClassification.from_pretrained("reward_model")

def score_response(prompt, response):
    text = prompt + " " + response
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True)
    with torch.no_grad():
        logits = model(**inputs).logits
        score = torch.softmax(logits, dim=1)[0][1].item()  # Probability of label 1 (thumbs up)
    return score

# Example usage
prompt = "Who is Bob Brown?"
response = "Bob Brown was a prolific scorer at the junior level in hockey."
print("Reward score:", score_response(prompt, response))


Reward score: 0.382489413022995


In [37]:
pip install trl transformers accelerate datasets


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 375.8/375.8 kB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.5/491.5 kB 21.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.6/193.6 kB 7.6 MB/s eta 0:00:00
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2025.3.2
    Uninstalling fsspec-2025.3.2:
      Successfully uninstalled fsspec-2025.3.2
  Attempting uninstall: datasets
    Found existing installation: datasets 2.14.4
    Uninstalling datasets-2.14.4:
      Successfully uninstalled datasets-2.14.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2025.3.2 requires fsspec==2025.3.2, but you have fsspec 2025.3.0 which is incompatible.


In [3]:
pip install -U trl


In [22]:
import trl
print(trl.__version__)


0.19.0


In [45]:
pip install --upgrade transformers


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 61.8 MB/s eta 0:00:00
  Attempting uninstall: transformers
    Found existing installation: transformers 4.52.4
    Uninstalling transformers-4.52.4:
      Successfully uninstalled transformers-4.52.4


In [2]:
from transformers import AutoModelForSequenceClassification, AutoTokenizer, pipeline

reward_tokenizer = AutoTokenizer.from_pretrained("reward_model")
reward_model = AutoModelForSequenceClassification.from_pretrained("reward_model")

reward_pipe = pipeline("text-classification", model=reward_model, tokenizer=reward_tokenizer, return_all_scores=True)


Device set to use cpu
/usr/local/lib/python3.11/dist-packages/transformers/pipelines/text_classification.py:106: UserWarning: `return_all_scores` is now deprecated,  if want a similar functionality use `top_k=None` instead of `return_all_scores=True` or `top_k=1` instead of `return_all_scores=False`.
  warnings.warn(


In [3]:
from transformers import AutoModelForCausalLM, AutoTokenizer

generator_model_name = "gpt2"
tokenizer = AutoTokenizer.from_pretrained(generator_model_name)
tokenizer.pad_token = tokenizer.eos_token  # GPT-2 needs this for padding

model = AutoModelForCausalLM.from_pretrained(generator_model_name)


In [10]:
import trl
print(trl.__version__)


0.19.0


In [5]:
pip install --upgrade trl


In [24]:
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForCausalLM, AutoModelForSequenceClassification
import pandas as pd

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

# Load your reward model and tokenizer (for scoring)
reward_tokenizer = AutoTokenizer.from_pretrained("reward_model")
reward_model = AutoModelForSequenceClassification.from_pretrained("reward_model").to(device)
reward_model.eval()  # set to eval mode since we're only scoring

# Load GPT-2 generator and tokenizer
gen_model_name = "gpt2"
tokenizer = AutoTokenizer.from_pretrained(gen_model_name)
tokenizer.pad_token = tokenizer.eos_token  # set pad token to eos_token
model = AutoModelForCausalLM.from_pretrained(gen_model_name).to(device)
model.train()

# Load your CSV and extract prompts (adjust column name accordingly)
df = pd.read_csv("cleaned_feedback.csv")
prompts = df["prompt"].dropna().tolist()

optimizer = torch.optim.Adam(model.parameters(), lr=5e-6)

def get_reward(prompt, response):
    text = prompt + " " + response
    # Tokenize with truncation to max 512 tokens (or your reward model's max length)
    encoding = reward_tokenizer(
        text,
        max_length=512,
        truncation=True,
        padding=True,
        return_tensors="pt"
    ).to(device)

    with torch.no_grad():
        outputs = reward_model(**encoding)
        logits = outputs.logits
        probs = torch.softmax(logits, dim=-1)
        # Assuming label 1 is the positive reward label; adjust as needed
        reward_score = probs[:, 1].item()
    return reward_score

epochs = 3
max_gen_length = 50

for epoch in range(epochs):
    print(f"Epoch {epoch+1}/{epochs}")
    for prompt in prompts:
        inputs = tokenizer(prompt, return_tensors="pt").to(device)

        # Generate tokens with sampling (to get varied responses)
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_gen_length,
            do_sample=True,
            top_k=50,
            top_p=0.95,
            pad_token_id=tokenizer.pad_token_id,
            return_dict_in_generate=True,
            output_scores=True,
        )

        sequences = outputs.sequences  # [batch_size=1, seq_len]

        # Decode generated tokens (excluding prompt tokens)
        generated_tokens = sequences[0, inputs.input_ids.shape[-1]:]
        response = tokenizer.decode(generated_tokens, skip_special_tokens=True)

        # Calculate reward with your reward model
        reward = get_reward(prompt, response)
        print(f"Prompt: {prompt}")
        print(f"Response: {response}")
        print(f"Reward: {reward:.4f}")

        # Calculate log probs of generated tokens
        logits = model(sequences[:, :-1]).logits  # predict next tokens
        log_probs = F.log_softmax(logits, dim=-1)

        # Gather log probs of generated tokens (teacher forcing)
        gen_token_ids = sequences[:, 1:]
        selected_log_probs = log_probs.gather(2, gen_token_ids.unsqueeze(-1)).squeeze(-1)

        # Sum log probs over generated tokens (excluding prompt tokens)
        gen_log_probs = selected_log_probs[0, inputs.input_ids.shape[-1]:].sum()

        # REINFORCE loss = -reward * log_prob
        loss = -reward * gen_log_probs

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()


Streaming output truncated to the last 5000 lines.
Answer:
Answer:

Answer:

Answer:

Answer:Answer:

Answer:

Answer:

Answer:

Answer:
Reward: 0.2307
Prompt: You are a helpful assistant. Use the following context to answer the question.

Context:
and work being an abridgment chiefly for the use of students of a life of a life of william shakespeare. london smith, elder co. ol 21113614m levenson, jill l. oxford oxford university press. isbn 978 0 19 281496 8. oclc 41991397 levin, harry 1986. critical approaches to shakespeare from 1660 to 1904. the cambridge companion to shakespeare studies. cambridge cambridge university press. isbn 978 0 521 31841 9. oclc 12945372 love, harold 2002. attributing authorship an introduction. cambridge cambridge university press. 1017 cbo9780511483165. isbn 978 0 511 48316 5. oclc 70741078 via cambridge core. shakespearean suspect texts the bad quartos and their contexts. cambridge cambridge university press. 1017 cbo9780511553134. isbn 978 0 511 55313 